In [16]:
import pandas as pd
import pyodbc
import nltk
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [17]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\krish\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [41]:
import pandas as pd
import pyodbc

def fetch_data_from_sql():

    conn_str = (
        'Driver={ODBC Driver 18 for SQL Server};'
        'Server=(local);'
        'Database=PortfolioProject_MarketingAnalytics;'
        'Trusted_Connection=yes;'
        'TrustServerCertificate=yes;'
    )

    conn = pyodbc.connect(conn_str)

    query = '''
    SELECT
        ReviewID,
        CustomerID,
        ProductID,
        ReviewDate,
        Rating,
        ReviewText
    FROM dbo.customer_reviews
    '''

    df = pd.read_sql(query, conn)

    conn.close()

    return df


customer_reviews_df = fetch_data_from_sql()

print(customer_reviews_df.head())

   ReviewID  CustomerID  ProductID  ReviewDate  Rating  \
0         1          77         18  2023-12-23       3   
1         2          80         19  2024-12-25       5   
2         3          50         13  2025-01-26       4   
3         4          78         15  2025-04-21       3   
4         5          64          2  2023-07-16       3   

                                 ReviewText  
0   Average  experience,  nothing  special.  
1            The  quality  is    top-notch.  
2   Five  stars  for  the  quick  delivery.  
3  Good  quality,  but  could  be  cheaper.  
4   Average  experience,  nothing  special.  


C:\Users\krish\AppData\Local\Temp\ipykernel_14424\855466033.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [42]:
customer_reviews_df = fetch_data_from_sql()

C:\Users\krish\AppData\Local\Temp\ipykernel_14424\855466033.py:27: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


In [43]:
import pyodbc

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client RDA 11.0', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server']


In [44]:
Driver={ODBC Driver 18 for SQL Server};

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2503285719.py, line 1)

In [45]:
sia = SentimentIntensityAnalyzer()

In [46]:
def calculate_sentiment(review):
    sentiment = sia.polarity_scores(review)
    return sentiment['compound']

In [47]:
sia = SentimentIntensityAnalyzer()

def calculate_sentiment(review):
    sentiment = sia.polarity_scores(review)
    return sentiment['compound']

In [48]:
def category_sentiment(score, rating):

    if score > 0.05:
        if rating >= 4:
            return 'Positive'
        elif rating == 3:
            return 'Mixed Positive'
        else:
            return 'Mixed Negative'
    elif score < -0.05:
        if rating <= 2:
            return 'Negative'
        elif rating == 3:
            return 'Mixed Negative'
        else:
            return 'Mixed Positive'
    else:
        if rating >= 4:
            return 'Positive'
        elif rating <= 2:
            return 'Negative'
        else:
            return 'Neutral'

In [57]:
# Define a function to bucket sentiment scores into text ranges
def sentiment_bucket(score):
    if score >= 0.5:
        return '0.5 to 1.0'
    elif 0.0 <= score < 0.5:
        return '0.0 to 0.49'
    elif -0.5 <= score < 0.0:
        return '-0.5 to -0.01'
    else:
        return '-1.0 to -0.5'

In [58]:
# Apply sentiment analysis to calculate sentiment scores for each review
customer_reviews_df['SentimentScore'] = customer_reviews_df['ReviewText'].apply(calculate_sentiment)

In [60]:
# Apply sentiment categorization using bith text and rating
customer_reviews_df['SentimentCategory'] = customer_reviews_df.apply(lambda row: category_sentiment(row['SentimentScore'], row['Rating']), axis=1)

In [62]:
customer_reviews_df['sentiment_bucket'] = customer_reviews_df['SentimentScore'].apply(sentiment_bucket)

In [63]:
print(customer_reviews_df.head())

   ReviewID  CustomerID  ProductID  ReviewDate  Rating  \
0         1          77         18  2023-12-23       3   
1         2          80         19  2024-12-25       5   
2         3          50         13  2025-01-26       4   
3         4          78         15  2025-04-21       3   
4         5          64          2  2023-07-16       3   

                                 ReviewText  SentimentScore SentimentCategory  \
0   Average  experience,  nothing  special.         -0.3089    Mixed Negative   
1            The  quality  is    top-notch.          0.0000          Positive   
2   Five  stars  for  the  quick  delivery.          0.0000          Positive   
3  Good  quality,  but  could  be  cheaper.          0.2382    Mixed Positive   
4   Average  experience,  nothing  special.         -0.3089    Mixed Negative   

  sentiment_bucket  
0    -0.5 to -0.01  
1      0.0 to 0.49  
2      0.0 to 0.49  
3      0.0 to 0.49  
4    -0.5 to -0.01  


In [64]:
customer_reviews_df.shape

(1363, 9)

In [66]:
customer_reviews_df.to_excel('fact_customer_reviews_with_sentiment.xlsx', index = False)